Dataset Source: [https://medic.renci.org/](https://medic.renci.org/)

In [1]:
import os
import pandas as pd
from collections import defaultdict
import json
from dataclasses import dataclass

In [2]:
@dataclass
class DrugRecord:
    name: str
    curie_label: str

@dataclass
class DiseaseRecord:
    label:      str
    definition: str | None
    synonyms:   str | None

@dataclass
class IndicationRecord:
    norm_drug_label:    str
    norm_disease_label: str
    disease_name:       str
    indications_text:   str | None
    hyperrelations:     list[dict] | None
    drug_record:        DrugRecord | None
    disease_record:     DiseaseRecord | None

class MedicMapping:
    drugs_df:       pd.DataFrame
    diseases_df:    pd.DataFrame
    indications_df: pd.DataFrame

    drug_code_to_rowidx:    dict[str, int]
    disease_code_to_rowidx: dict[str, int]

    drug_indication_rowmap:               defaultdict[str, list[int]]
    disease_indication_rowmap:            defaultdict[str, list[int]]
    normalized_disease_indication_rowmap: defaultdict[str, list[int]]

    def __init__(self, data_dir: str) -> None:
        self.drugs_df       = pd.read_csv(os.path.join(data_dir, 'MeDIC Drug List.csv'))
        self.diseases_df    = pd.read_csv(os.path.join(data_dir, 'Diseases List.csv'))
        self.indications_df = pd.read_csv(os.path.join(data_dir, 'Indications List.csv'))

        self.drug_code_to_rowidx    = dict()
        self.disease_code_to_rowidx = dict()

        self.drug_indication_rowmap               = defaultdict(list)
        self.disease_indication_rowmap            = defaultdict(list)
        self.normalized_disease_indication_rowmap = defaultdict(list)

        for idx, row in self.drugs_df.iterrows():
            code = str(row['curie']).lower().strip()
            assert code not in self.drug_code_to_rowidx
            self.drug_code_to_rowidx[code] = int(idx) # type: ignore

        for idx, row in self.diseases_df.iterrows():
            code = str(row['category_class']).lower().strip()
            assert code not in self.disease_code_to_rowidx
            self.disease_code_to_rowidx[code] = int(idx) # type: ignore

        for idx, row in self.indications_df.iterrows():
            drug         = str(row['final normalized drug label']).lower().strip()
            disease      = str(row['disease name']).lower().strip()
            norm_disease = str(row['final normalized disease label']).lower().strip()
            idx = int(idx) # type: ignore
            self.drug_indication_rowmap[drug].append(idx)
            self.disease_indication_rowmap[disease].append(idx)
            self.normalized_disease_indication_rowmap[norm_disease].append(idx)

    def load_drug_record(self, code: str | None):
        if code is None:
            return None

        rowidx = self.drug_code_to_rowidx.get(code)
        if rowidx is None:
            return None

        row = self.drugs_df.iloc[rowidx]

        return DrugRecord(name = row['drug_name'], curie_label = row['curie_label'])

    def load_disease_record(self, code: str | None):
        if code is None:
            return None

        rowidx = self.disease_code_to_rowidx.get(code)
        if rowidx is None:
            return None

        row = self.diseases_df.iloc[rowidx]

        return DiseaseRecord(label = row['label'], definition = row['definition'], synonyms = row['synonyms'])

    def load_indication_by_idx(self, rowidx: int):
        row = self.indications_df.iloc[rowidx]

        norm_drug_label    = row['final normalized drug label']
        norm_disease_label = row['final normalized disease label']
        disease_name       = row['disease name']
        indications_text   = row['indications text']

        drug_code    = str(row['final normalized drug id']).lower().strip()
        disease_code = str(row['final normalized disease id']).lower().strip()

        drug_record    = self.load_drug_record(drug_code)
        disease_record = self.load_disease_record(disease_code)

        hyperrelations = row['hyperrelations']

        if hyperrelations is not None and isinstance(hyperrelations, str):
            hyperrelations = hyperrelations.strip()

            if hyperrelations.startswith('```json') and hyperrelations.endswith('```'):
                try:
                    hyperrelations = json.loads(hyperrelations[7:-3])
                except:
                    hyperrelations = None
            else:
                hyperrelations = None

        return IndicationRecord(
            norm_drug_label    = norm_drug_label,
            norm_disease_label = norm_disease_label,
            disease_name       = disease_name,
            indications_text   = indications_text,
            hyperrelations     = hyperrelations,
            drug_record        = drug_record,
            disease_record     = disease_record
        )

    def find_indications_by_disease(self, name: str, normalized: bool):
        ddict = self.disease_indication_rowmap if not normalized else self.normalized_disease_indication_rowmap

        name = name.lower().strip()

        if name not in ddict:
            return []

        row_indices = ddict[name]

        return [self.load_indication_by_idx(i) for i in row_indices]

    def find_indications_by_drug(self, name: str):
        ddict = self.drug_indication_rowmap

        name = name.lower().strip()

        if name not in ddict:
            return []

        row_indices = ddict[name]

        return [self.load_indication_by_idx(i) for i in row_indices]


In [3]:
mdmp = MedicMapping('./dataset')

In [4]:
indications = mdmp.find_indications_by_disease('cerebral palsy', normalized = True)

indications

[IndicationRecord(norm_drug_label='Dantrolene', norm_disease_label='cerebral palsy', disease_name='cerebral palsy', indications_text="In Chronic Spasticity: Dantrium is indicated in controlling the manifestations of clinical spasticity resulting from upper motor neuron disorders (e.g., spinal cord injury, stroke, cerebral palsy, or multiple sclerosis). It is of particular benefit to the patient whose functional rehabilitation has been retarded by the sequelae of spasticity. Such patients must have presumably reversible spasticity where relief of spasticity will aid in restoring residual function. Dantrium is not indicated in the treatment of skeletal muscle spasm resulting from rheumatic disorders. If improvement occurs, it will ordinarily occur within the dosage titration (see DOSAGE AND ADMINISTRATION ), and will be manifested by a decrease in the severity of spasticity and the ability to resume a daily function not quite attainable without Dantrium. Occasionally, subtle but meaningf